# Using the Renewable Component Library (RCL)

This notebook demonstrates how to search and download PV modules and inverters from DNV's Renewable Component Library using the SolarFarmer Python SDK.

See details in SolarFarmer's public documentation:
- [RCL Endpoint](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/RclApi/RclEndpoint.html)
- [RCL Tutorial](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/RclApi/RclEndpointTutorial.html)

## 0. Prerequisites

**Notebook Information:**
- **Last Updated:** August 2026
- **Written for:** SolarFarmer SDK v0.5.0+
- [View latest version in repository](https://github.com/dnv-opensource/solarfarmer-python-sdk/blob/main/docs/notebooks/Example_RCL_Catalog.ipynb)

### 0.1 Install the SolarFarmer Python SDK

This notebook requires the SolarFarmer Python SDK to be installed. Install it via pip:

```bash
pip install dnv-solarfarmer
```

In [1]:
import os
import solarfarmer as sf

# Check SDK version compatibility
NOTEBOOK_MIN_SDK_VERSION = "0.5.0"

print(f"SolarFarmer Python SDK v{sf.__version__}")

# Parse versions for comparison
def parse_version(v):
    """Simple version parser for x.y.z format"""
    return tuple(map(int, v.split('.')))

try:
    if parse_version(sf.__version__) < parse_version(NOTEBOOK_MIN_SDK_VERSION):
        print(f"\n    WARNING: This notebook requires SDK v{NOTEBOOK_MIN_SDK_VERSION} or later.")
        print(f"    Your version: {sf.__version__}")
        print(f"    Some examples may not work correctly.")
        print(f"    Upgrade with: pip install --upgrade dnv-solarfarmer\n")
except Exception:
    pass

sf.configure_logging()

SolarFarmer Python SDK v0.5.0


<Logger solarfarmer (INFO)>

### 0.2 API Key Required

You need a SolarFarmer API key to access the RCL. Instructions for acquiring one is [HERE](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/Introduction/ApiKey.html)

**Important:** Avoid hardcoding your API key directly in notebook cells.

- **Use environment variables (Recommended):**

  The SDK automatically uses the `SF_API_KEY` environment variable. Set it in your terminal before starting Jupyter:

  **Linux/Mac:**
  ```bash
  export SF_API_KEY="your-key-here"
  ```

  **Windows:**
  ```bash
  set SF_API_KEY=your-key-here
  ```

- **Entering your API key (Alternative):**

  This notebook will prompt you to enter your API key and keep it hidden from view.

In [2]:
if os.getenv("SF_API_KEY") is None:
    print("WARNING: `SF_API_KEY` environment variable not set.")
    import getpass
    api_key = getpass.getpass("Enter your SolarFarmer API key: ")
    print("Using API key entered by user.\n")
else:
    api_key = os.getenv("SF_API_KEY")
    print("Using API key from environment variable `SF_API_KEY`")

Using API key from environment variable `SF_API_KEY`


## 1. Rate Limit Status

Before downloading files, check your remaining monthly quota. This call is **free** - it doesn't consume any downloads.

In [3]:
status = sf.rcl.get_rate_limit_status(api_key=api_key)

print(f"Downloads remaining: {status.remaining}/{status.limit}")
print(f"Usage: {status.usage_percent:.1f}%")
print(f"Resets: {status.reset_datetime}")

if status.is_low:
    print("\n⚠️ Warning: Running low on downloads!")

Downloads remaining: 93/100
Usage: 7.0%
Resets: 2026-09-04 00:00:00+00:00


## 2. Searching for PV Modules

Use `sf.rcl.list_modules()` to search the module catalog. You can filter by manufacturer, model, power, and other attributes. 

See [Section 7](#filter-reference) for a complete list of available module filters.

### 2.1 Basic Module Search

Just filtering a desired number of modules (via `top` parameter). Default is 25 records. The maximum value is 10,000 records.

In [4]:
%%time
# Get first 5 modules from the catalog
result = sf.rcl.list_modules(top=5, api_key=api_key)

print(f"Total modules in catalog: {result['total']}")
print(f"\nFirst {len(result['items'])} modules:")

for item in result["items"]:
    print(f"  - {item['manufacturer']} {item['model']}")

INFO: 1197 modules found, retrieved 5.
Total modules in catalog: 1197

First 5 modules:
  - AE Solar AE MD-144 530
  - AE Solar AE MD-144 535
  - AE Solar AE MD-144 540
  - AE Solar AE MD-144 545
  - AE Solar AE MD-144 550
CPU times: total: 31.2 ms
Wall time: 1.34 s


### 2.2 Filtered Module Search

Filter modules by manufacturer, power range, and other attributes.

See [Section 7](#filter-reference) for a complete list of available inverter filters.


In [5]:
# Search for high-power modules from Canadian Solar
result = sf.rcl.list_modules(
    manufacturer_contains="Canadian",
    p_nom_gte=600,  # Minimum 600W
    order_by="pNom",
    order_dir="DESC",
    top=10,
    api_key=api_key,
)

print(f"\nTop 10 by power:")

for item in result["items"]:
    power = item.get('pNom', 'N/A')
    print(f"  - {item['manufacturer']} {item['model']}: {power}W")

INFO: 45 modules found, retrieved 10.

Top 10 by power:
  - Canadian Solar CS7N-730TB-AG: 730.0W
  - Canadian Solar CS7N-725TB-AG: 725.0W
  - Canadian Solar CS7N-720TB-AG: 720.0W
  - Canadian Solar CS7N-715TB-AG: 715.0W
  - Canadian Solar CS7N-710TB-AG: 710.0W
  - Canadian Solar CS7N-705TB-AG: 705.0W
  - Canadian Solar CS7N-700TB-AG: 700.0W
  - Canadian Solar CS7N-695TB-AG: 695.0W
  - Canadian Solar CS7N-690TB-AG: 690.0W
  - Canadian Solar CS7N-685TB-AG: 685.0W


### 2.3 Search for Bifacial Modules


In [6]:
# Search for bifacial modules (bifaciality factor >= 0.7)
result = sf.rcl.list_modules(
    manufacturer_contains="LONGi",
    bifaciality_factor_gte=0.7,
    p_nom_gte=550,
    output_parameter=["pNom", "bifacialityFactor"],
    top=5,
    api_key=api_key,
)

INFO: 58 modules found, retrieved 5.


To get information about each of the resulting components, note that these can be queried either via typed properties with the `RCLCatalogItem` class or via their dictionary key.

In [7]:
# Using RCLCatalogItem for better IDE support
from solarfarmer import RCLCatalogItem

print("Using RCLCatalogItem (useful for IDE autocomplete):")
for raw_item in result["items"]:
    item = RCLCatalogItem(raw_item)  # Wrap in typed class
    print(f"  - {item.manufacturer} {item.model}: {item.p_nom}W, BF={item.bifaciality_factor}")

print("\nUsing dict-style access:")
for item in result["items"]:
    print(f"  - {item['manufacturer']} {item['model']}: {item.get('pNom')}W, BF={item.get('bifacialityFactor')}")

Using RCLCatalogItem (useful for IDE autocomplete):
  - LONGi LR5-72HGD-560M: 560.0W, BF=0.8
  - LONGi LR5-72HGD-565M: 565.0W, BF=0.8
  - LONGi LR5-72HGD-570M: 570.0W, BF=0.8
  - LONGi LR5-72HGD-575M: 575.0W, BF=0.8
  - LONGi LR5-72HGD-580M: 580.0W, BF=0.8

Using dict-style access:
  - LONGi LR5-72HGD-560M: 560.0W, BF=0.8
  - LONGi LR5-72HGD-565M: 565.0W, BF=0.8
  - LONGi LR5-72HGD-570M: 570.0W, BF=0.8
  - LONGi LR5-72HGD-575M: 575.0W, BF=0.8
  - LONGi LR5-72HGD-580M: 580.0W, BF=0.8


## 3. Searching for Inverters

Use `sf.rcl.list_inverters()` to search the inverter catalog. You can filter by power, efficiency, MPPT voltage range, and more. 

See [Section 7](#filter-reference) for a complete list of available inverter filters.

### 3.1 Basic Inverter Search


In [8]:
# Get first 5 inverters from the complete catalog
result = sf.rcl.list_inverters(top=5, api_key=api_key)

print(f"\nFirst {len(result['items'])} inverters:")

for item in result["items"]:
    print(f"  - {item['manufacturer']} {item['model']}")

INFO: 318 inverters found, retrieved 5.

First 5 inverters:
  - ABB Proteus PV 4100
  - ABB Proteus PV 4300
  - ABB Proteus PV 4500
  - ABB Proteus PV 4700
  - Chint CPS SCA100K-T-US-480


### 3.2 Filtered Inverter Search

Note the INFO message about the query can be turned off via the parameter `verbose=False`

In [9]:
# Search for high-efficiency utility-scale inverters
result = sf.rcl.list_inverters(
    manufacturer_contains="Sungrow",
    p_nom_conv_gte=200,  # Min 200 kW
    effic_max_gte=98.5,  # Min 98.5% efficiency
    top=10,
    api_key=api_key,
    verbose=False, # Set to True to print total number of items and those retrieved up to top pagination limit.
)

print(f"\nFound {result['total']} inverters matching criteria:")

for item in result["items"]:
    power = item.get('pNomConv', 'N/A')
    eff = item.get('efficMax', 'N/A')
    print(f"  - {item['manufacturer']} {item['model']}: {power}kW, {eff}% max eff")


Found 5 inverters matching criteria:
  - Sungrow SC2750UD-MV-US: 2750.0kW, 99.0% max eff
  - Sungrow SC3150UD-MV-US: 3150.0kW, 99.0% max eff
  - Sungrow SC3450UD-MV-US: 3450.0kW, 99.0% max eff
  - Sungrow SG320HX: 320.0kW, 99.02% max eff
  - Sungrow SG320HX-20: 320.0kW, 99.02% max eff


### 3.3 Filter by MPPT Voltage Range

In [10]:
# Find inverters compatible with high-voltage strings
result = sf.rcl.list_inverters(
    v_mpp_max_gte=1100,  # MPPT range extends to at least 1100V
    nb_mppt_gte=10,      # At least 10 MPPTs
    output_parameter=["pNomConv", "vMppMin", "vMppMax", "nbMppt"],
    top=5,
    api_key=api_key,
    verbose=False, # Set to True to print total number of items and those retrieved up to top pagination limit.
)

print(f"Found {result['total']} high-voltage inverters")

for item in result["items"]:
    vmin = item.get('vMppMin', 'N/A')
    vmax = item.get('vMppMax', 'N/A')
    mppts = item.get('nbMppt', 'N/A')
    print(f"  - {item['manufacturer']} {item['model']}: {vmin}-{vmax}V, {mppts} MPPTs")

Found 22 high-voltage inverters
  - Chint CPS SCH275KTL-DO/US-800-24 V2: 900-1300V, 12 MPPTs
  - Chint CPS SCH275KTL-DO/US-800-36 V2: 900-1300V, 12 MPPTs
  - Chint SCH333K-T-EU: 500-1500V, 15 MPPTs
  - Chint SCH350K-T-EU: 500-1500V, 15 MPPTs
  - Chint SCH350KTL-DO/US-800: 880-1300V, 15 MPPTs


## 4. Pagination

For large result sets, use `top` and `skip` to paginate through results.

In [11]:
# Example: Iterate through all Trina modules
all_items = []
skip = 0
page_size = 100
max_pages = 3  # Limit for demo purposes

for page in range(max_pages):
    result = sf.rcl.list_modules(
        manufacturer_contains="Trina",
        top=page_size,
        skip=skip,
        api_key=api_key,
    )
    all_items.extend(result["items"])
    
    print(f"Page {page + 1}: Retrieved {len(result['items'])} items (skip={skip})")
    
    if len(result["items"]) < page_size:
        break  # Last page
    skip += page_size

print(f"\nTotal retrieved: {len(all_items)} modules")

INFO: 91 modules found, retrieved 91.
Page 1: Retrieved 91 items (skip=0)

Total retrieved: 91 modules


## 5. Downloading Files

⚠️ **Warning:** Each download counts against your monthly quota. Check your rate limit status before downloading.

The cells below are set to not run by default to preserve your download quota. Remove the `if False:` guard to execute them.

In [12]:
# Check rate limit before downloading
status = sf.rcl.get_rate_limit_status(api_key=api_key)
print(f"Downloads remaining: {status.remaining}/{status.limit}")

if status.remaining < 5:
    print("\n⚠️ Low on downloads. Consider waiting until reset.")
    print(f"   Resets: {status.reset_datetime}")

Downloads remaining: 93/100


In [ ]:
# Example: Download a module file (DISABLED by default - remove 'if False:' to run)
if False:
    # First, find a specific module
    result = sf.rcl.list_modules(
        manufacturer="Canadian Solar",
        model_contains="CS7N-715",
        top=1,
        api_key=api_key,
    )

    if result["items"]:
        item = RCLCatalogItem(result["items"][0])
        print(f"Downloading: {item.filename}")
        
        # Download to current directory
        content = sf.rcl.download_file(
            item.file_uuid,
            item.filename,
            directory_path="./rcl_downloads/",
            api_key=api_key,
        )
        
        print(f"Downloaded {len(content)} bytes")
        print(f"File saved as: {item.filename}")

INFO: 1 modules found, retrieved 1.
Downloading: CanadianSolar_CS7N-715TB-AG.PAN


INFO: Saved RCL file to rcl_downloads\CanadianSolar_CS7N-715TB-AG.PAN


Downloaded 841 bytes
File saved as: CanadianSolar_CS7N-715TB-AG.PAN


### 5.1 Handling duplicated download queries: cached files

If you run the same download (same file name and directory), the `download_file` function will first check that the file does not exist in the folder. If it exists, the download will be skipped to protect you from wasting your quota to download the same file again (e.g., running a workflow in a loop).

You can force the download by setting the parameter `use_cache=False`, this property's default is `True`.


In [ ]:
# Example: Download a module file (DISABLED by default - remove 'if False:' to run)
if False:
    # First, find a specific module
    result = sf.rcl.list_modules(
        manufacturer="Canadian Solar",
        model_contains="CS7N-715",
        top=1,
        api_key=api_key,
    )
    print(f"{result['total']} modules found, retrieved {len(result['items'])}.")

    if result["items"]:
        item = RCLCatalogItem(result["items"][0])
        print(f"Downloading: {item.filename}")
    # Download to current directory the same module as above
    content = sf.rcl.download_file(
        item.file_uuid,
        item.filename,
        directory_path="./rcl_downloads/",
        api_key=api_key,
    )

INFO: Using cached file: rcl_downloads\CanadianSolar_CS7N-715TB-AG.PAN (skipping download)


### 5.2 Hanlding incorrect download queries

Note the typo below (i.e., *Canad**ai**n* instead of *Canad**ia**n*) in the manufacturer name. 

When there are no matches, an info message will indicate that the 0 modules were found.

In [ ]:
# Example: Download a module file (DISABLED by default - remove 'if False:' to run)
if False:
    # First, find a specific module
    result = sf.rcl.list_modules(
        manufacturer="Canadain Solar", # Note the typo here, this will return no results
        model_contains="CS7N-715",
        top=1,
        api_key=api_key,
    )

INFO: 0 modules found, retrieved 0.


### 6. PVSystem Integration

`PVSystem` provides convenience methods `set_module_from_rcl()` and `set_inverter_from_rcl()` that combine the search and download steps into a single call. If the search returns exactly one match, the file is downloaded and assigned to the plant; if multiple matches are found, a `ValueError` is raised listing the candidates so you can refine your filters.

⚠️ **Note:** These methods download files and consume your monthly quota. Caching applies — re-running with the same filters and directory will reuse the local file.


In [16]:
# Create a PVSystem instance
plant = sf.PVSystem(
    name="RCL Demo Plant",
    latitude=33.45,
    longitude=-112.07,
    dc_capacity_MW=10.0,
    ac_capacity_MW=8.0,
    mounting="Fixed",
)

print(f"Created plant: {plant.name}")
print(f"Current PAN files: {plant.pan_files}")
print(f"Current OND files: {plant.ond_files}")

Created plant: RCL Demo Plant
Current PAN files: {}
Current OND files: {}


In [ ]:
# Example: Set module from RCL (DISABLED by default - remove 'if False:' to run)
if False:
    module = plant.set_module_from_rcl(
        manufacturer_contains="Canadian Solar",
        model_contains="CS7N-725TB-AG",
        directory_path="./rcl_downloads/",
        api_key=api_key,
        strict=False,  # prints suggestions instead of raising on no/multiple matches
    )
    if module:
        print(f"Module assigned: {module['filename']}")
        print(f"PAN files: {plant.pan_files}")

INFO: 1 modules found, retrieved 1.


INFO: Saved RCL file to rcl_downloads\CanadianSolar_CS7N-725TB-AG.PAN
INFO: Module 'CanadianSolar_CS7N-725TB-AG' set from RCL (841 bytes)


Module assigned: CanadianSolar_CS7N-725TB-AG.PAN
PAN files: {'CanadianSolar_CS7N-725TB-AG': WindowsPath('rcl_downloads/CanadianSolar_CS7N-725TB-AG.PAN')}


In [ ]:
# Example: Set inverter from RCL (DISABLED by default - remove 'if False:' to run)
if False:
    inverter = plant.set_inverter_from_rcl(
        manufacturer_contains="Sungrow",
        model_contains="SG250HX",
        directory_path="./rcl_downloads/",
        api_key=api_key,
        strict=False,  # prints suggestions instead of raising on no/multiple matches
    )
    if inverter:
        print(f"Inverter assigned: {inverter['filename']}")
        print(f"OND files: {plant.ond_files}")


INFO: 1 inverters found, retrieved 1.


INFO: Saved RCL file to rcl_downloads\Sungrow_SG250HX_800V.OND
INFO: Inverter 'Sungrow_SG250HX_800V' set from RCL (2157 bytes)


Inverter assigned: Sungrow_SG250HX_800V.OND
OND files: {'Sungrow_SG250HX_800V': WindowsPath('rcl_downloads/Sungrow_SG250HX_800V.OND')}


### 6.1 Handling Multiple Component Matches

When `set_module_from_rcl()` finds more than one result, it raises a `ValueError` listing the candidates instead of silently picking one (default `strict=True`). Pass `strict=False` to print the suggestions and return `None` instead — useful during exploration or for users who prefer not to handle exceptions explicitly.

**Example with suggestions printed out**

In [19]:
# Broad search — strict=False prints candidates instead of raising ValueError
module = plant.set_module_from_rcl(
    manufacturer_contains="Canadian Solar",
    model_contains="CS7N",           # too broad: matches many models
    directory_path="./rcl_downloads/",
    api_key=api_key,
    strict=False, # prints suggestions instead of raising an error on no/multiple matches
)

INFO: 12 modules found, retrieved 12.
INFO: 12 modules found. Narrow your search:

  1. Canadian Solar - CS7N-675TB-AG (675W bifacial)
     → Add: model="CS7N-675TB-AG" or p_nom=675 or bifaciality_factor_gte=0.7

  2. Canadian Solar - CS7N-680TB-AG (680W bifacial)
     → Add: model="CS7N-680TB-AG" or p_nom=680 or bifaciality_factor_gte=0.7

  3. Canadian Solar - CS7N-685TB-AG (685W bifacial)
     → Add: model="CS7N-685TB-AG" or p_nom=685 or bifaciality_factor_gte=0.7

  4. Canadian Solar - CS7N-690TB-AG (690W bifacial)
     → Add: model="CS7N-690TB-AG" or p_nom=690 or bifaciality_factor_gte=0.7

  5. Canadian Solar - CS7N-695TB-AG (695W bifacial)
     → Add: model="CS7N-695TB-AG" or p_nom=695 or bifaciality_factor_gte=0.7

  6. Canadian Solar - CS7N-700TB-AG (700W bifacial)
     → Add: model="CS7N-700TB-AG" or p_nom=700 or bifaciality_factor_gte=0.7

  7. Canadian Solar - CS7N-705TB-AG (705W bifacial)
     → Add: model="CS7N-705TB-AG" or p_nom=705 or bifaciality_factor_gte=0.7

  8. Ca

**Example with `ValueError` raised**

This is the default behaviour when the parameter `strict` is not passed.

In [20]:
# Broad search — strict=True to raise ValueError
module = plant.set_module_from_rcl(
    manufacturer_contains="Canadian Solar",
    model_contains="CS7N",           # too broad: matches many models
    directory_path="./rcl_downloads/",
    api_key=api_key,
    strict=True, # raises an error on no/multiple matches
)

INFO: 12 modules found, retrieved 12.


ValueError: 12 modules found. Narrow your search:

  1. Canadian Solar - CS7N-675TB-AG (675W bifacial)
     → Add: model="CS7N-675TB-AG" or p_nom=675 or bifaciality_factor_gte=0.7

  2. Canadian Solar - CS7N-680TB-AG (680W bifacial)
     → Add: model="CS7N-680TB-AG" or p_nom=680 or bifaciality_factor_gte=0.7

  3. Canadian Solar - CS7N-685TB-AG (685W bifacial)
     → Add: model="CS7N-685TB-AG" or p_nom=685 or bifaciality_factor_gte=0.7

  4. Canadian Solar - CS7N-690TB-AG (690W bifacial)
     → Add: model="CS7N-690TB-AG" or p_nom=690 or bifaciality_factor_gte=0.7

  5. Canadian Solar - CS7N-695TB-AG (695W bifacial)
     → Add: model="CS7N-695TB-AG" or p_nom=695 or bifaciality_factor_gte=0.7

  6. Canadian Solar - CS7N-700TB-AG (700W bifacial)
     → Add: model="CS7N-700TB-AG" or p_nom=700 or bifaciality_factor_gte=0.7

  7. Canadian Solar - CS7N-705TB-AG (705W bifacial)
     → Add: model="CS7N-705TB-AG" or p_nom=705 or bifaciality_factor_gte=0.7

  8. Canadian Solar - CS7N-710TB-AG (71

**Example with correct (refined / narrowed down) search**

Provide a specific model to get a desired instance. 

*Note: cached from local disk*

In [ ]:
# Refined search — specific enough to resolve to a single match (DISABLED by default - remove 'if False:' to run)
if False:
    module = plant.set_module_from_rcl(
        manufacturer_contains="Canadian Solar",
        model_contains="CS7N-715TB-AG",  # specific enough for a single result
        directory_path="./rcl_downloads/",
        api_key=api_key,
        strict=False,
    )
    if module:
        print(f"Module assigned: {module['filename']}")
        print(f"PAN files: {plant.pan_files}")

## 7. Filter Reference

### Module Filters

| Parameter | Description | Example |
|-----------|-------------|---------|
| `manufacturer` | Exact manufacturer match | `"Canadian Solar"` |
| `manufacturer_contains` | Manufacturer contains substring | `"Canadian"` |
| `model` | Exact model match | `"CS7N-715TB-AG"` |
| `model_contains` | Model contains substring | `"715TB"` |
| `p_nom_gte` | Minimum nominal power (W) | `600` |
| `p_nom_lte` | Maximum nominal power (W) | `800` |
| `bifaciality_factor_gte` | Minimum bifaciality factor | `0.7` |
| `technol` | Technology type | `"mtSi"` |
| `lifecycle_status` | Lifecycle status | `"active"` |

### Inverter Filters

| Parameter | Description | Example |
|-----------|-------------|---------|
| `manufacturer` | Exact manufacturer match | `"Sungrow"` |
| `manufacturer_contains` | Manufacturer contains substring | `"Sungrow"` |
| `model` | Exact model match | `"SG250HX"` |
| `model_contains` | Model contains substring | `"HX"` |
| `p_nom_conv_gte` | Min rated AC power (kW) | `100` |
| `effic_max_gte` | Min max efficiency (%) | `98.5` |
| `v_mpp_min_lte` | Max lower MPPT voltage (V) | `500` |
| `v_mpp_max_gte` | Min upper MPPT voltage (V) | `1100` |
| `nb_mppt_gte` | Min number of MPPTs | `12` |

### Pagination & Ordering

| Parameter | Default | Description |
|-----------|---------|-------------|
| `top` | 25 | Page size (max 10000) |
| `skip` | 0 | Offset for pagination |
| `order_by` | - | Field to sort by |
| `order_dir` | ASC | Sort direction: ASC or DESC |
| `output_parameter` | all | List of fields to return |

## Next Steps

- See [RCL Documentation](../getting-started/rcl-component-library.md) for detailed usage via the SDK
- Explore [Workflow 2](../getting-started/workflow-2-pvplant-builder.md) to use downloaded files with PVSystem
- Check the [API Reference](../api.md#renewable-component-library-rcl) for full function signatures
- If you need details on the API endpoint, visit [rcl endpoint](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/RclApi/RclEndpoint.html) in the SolarFarmer public documentation
- There is also a basic endpoint tutorial in [rcl endpoint tutorial](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/RclApi/RclEndpointTutorial.html)